<a href="https://colab.research.google.com/github/Valementajat/PageRank_colab/blob/master/pageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
pip install kaggle

In [7]:
import os
os.environ['KAGGLE_USERNAME'] = "johannestakamki"
os.environ['KAGGLE_KEY'] = "KGAT_3f821097959746ff4db1bbb5db1fa0f9"

dataset_zip = "new-york-times-articles-comments-2020.zip"
if not os.path.exists(dataset_zip):
  !kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020


In [8]:
# Unzip the file independenty
articles = "nyt-articles-2020.csv"
comments = "nyt-comments-2020.csv"

if not os.path.exists(articles) and not os.path.exists(comments):
  !unzip new-york-times-articles-comments-2020.zip

In [25]:
import numpy as np
import pandas as pd

# importing random for making a subsample of the dataset
import random

from collections import defaultdict, Counter
import gc

In [10]:
chunk_size = 10000

ids = []


# To guard against the memory we load the csv in chuncks and process it right away
for article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):
   for article_id in article_chunk["uniqueID"]:
      if article_id not in ids:
              ids.append(article_id)
n = len(ids)

In [11]:
# Lets create the dataset we want to work from the articles
sample_frac = 0.2
sample_size = int(n * sample_frac)

sample_ids = random.sample(ids, sample_size)
n = len(sample_ids)
# Maybe this will be a bit more efficient to work with than a full on URL?
id_to_i = {article_id: i for i, article_id in enumerate(sample_ids)}



In [12]:
# Creating an empty adjancency matrix
adjencencyMatrix =  np.zeros((n, n ), dtype=float)



In [13]:
# First part of the mapReduce, we create the key value pairs
KeyValuePairs = []

In [14]:
# TO avoid blowing up the memory usage, we load the
# comments in chucks
chunk_size = 10000
for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):
  # Start the MapReduce by creating key-value pairs
  for index, row in chunk.iterrows():
    if row['articleID'] in id_to_i:
      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))

del chunk
del chunk_size

gc.collect()



0

In [15]:
# Lets group the KeyValuePairs by the key
KeyValueGroup = defaultdict(set)
for article_idx, user_id in KeyValuePairs:
    KeyValueGroup[article_idx].add(user_id)


In [16]:
# Then the final reduce so we can use this
connectionPairs = []

items = list(KeyValueGroup.items())

# This is a possibly a problem?
# I'm checking items against other values in the same set and looking if there
# are two unique intersecting user pairs so we can say that there
for a_idx, users_a in items:
    for b_idx, users_b in items:
        if a_idx != b_idx:
            if len(users_a.intersection(users_b)) >= 2:
                connectionPairs.append((a_idx, b_idx))


In [32]:
for i in range(10):
  print(connectionPairs[i])

(471, 3080)
(471, 737)
(471, 319)
(471, 2882)
(471, 2517)
(471, 2157)
(471, 3112)
(471, 3193)
(471, 2077)
(471, 2511)


In [33]:
# degree is practically how many times that page is seen in the connections
degree = Counter()
for i, j in connectionPairs:
    degree[i] += 1


for i, j in connectionPairs:
    adjencencyMatrix[i, j] = 1.0 / degree[i]


In [34]:
#We have a problem, dangling nodes, so this is to uniformally
# set the propability to go from a page to every page when calculating the rank

row_sums = adjencencyMatrix.sum(axis=1)
dangling = (row_sums == 0)


# replace each dangling row with uniform distribution
adjencencyMatrix[dangling, :] = 1.0 / n

In [35]:
# So we have our stochastic adjencencyMatrix or connection matrix

# Without any dangling pages (or nodes)

# Lets run the power iteration

# This is the formula to "teleport", during the power iteration.
# as in, do we take the next page from the page's links or do we take one from
# all of the pages
# P = βM + (1 - β)1/N

beta = 0.85 # example size is between 0.8 - 0.9
n = len(adjencencyMatrix)

#now this is the pageRank transition matrix or the google matrix
P = beta * adjencencyMatrix + (1 - beta) / n

In [46]:
# lets initialize the rank vector r^(0) which has every rank as 1/n
r = np.ones(n) / n

In [48]:
# Now we calculat the new ranks using the transition matrix and the old rank
# r^new =  r^old · A
for i in range(100):
  # python's matrix manipulation to multiply matrix with another matrix
    r = r @ P

In [49]:
# This is just a sanity check
print(r.sum())

1.0000000000000013


In [ ]:
# Top 10 indices by rank (largest first)
top_k = 10
top_idx = np.argsort(r)[-top_k:][::-1]

print("Top ranks:")
for idx in top_idx:
    print(idx, r[idx])

In [ ]:
%whos